In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
import os
import json

from dotenv import load_dotenv
load_dotenv()

import ollama

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.tools import tool
from langgraph.prebuilt import InjectedState

from file_tools import (
    DeepAgentState,
    ls,
    read_file,
    write_file,
    cleanup_files,
)
from prompts import (
    ORCHESTRATOR_PROMPT,
    RESEARCHER_PROMPT,
    EDITOR_PROMPT,
)

In [3]:
# -------------------------
# Web Search Tool
# -------------------------

@tool
def web_search(query: str) -> str:
    """
    Perform a live web search using Ollama Cloud Web Search API.

    Input:
        query: search query string

    Output:
        JSON string of top results (max_results=2).
    """
    response = ollama.web_search(query, max_results=2)
    response = response.results

    return response


In [4]:
result = web_search.invoke({"query": "Latest global news"})

In [5]:
result

[WebSearchResult(content='U.S. launches strikes in Syria targeting Islamic State fighters : NPR\nAccessibility links\n* [Skip to main content](#mainContent)\n* [Keyboard shortcuts for audio player](https://help.npr.org/contact/s/article?name=what-are-the-keyboard-shortcuts-for-using-the-npr-org-audio-player)\n**U.S. launches strikes in Syria targeting Islamic State fighters****The Trump administration launched military strikes in Syria to "eliminate" Islamic State group fighters in retaliation for an attack that killed two U.S. troops and an American interpreter a week ago.**\n### [National Security](https://www.npr.org/sections/national-security/)\n# U.S. launches strikes in Syria targeting Islamic State fighters after American deaths\nDecember 19, 202511:18 PM ET\nBy\n[The Associated Press](https://www.npr.org/people/101453150/the-associated-press)\n![President Donald Trump and Defense Secretary Pete Hegseth salute as carry teams move the transfer cases with the remains of Iowa Natio

In [6]:
# -------------------------
# LLM
# -------------------------

llm = ChatGoogleGenerativeAI(model="gemini-3-flash-preview")
# llm = ChatGoogleGenerativeAI(model="gemini-3-pro-preview")

In [7]:
# -------------------------
# Worker Agents: Researcher & Editor
# -------------------------
researcher_tools = [ls, write_file, read_file, web_search]

researcher_agent = create_agent(
    model=llm,
    tools=researcher_tools,
    system_prompt=RESEARCHER_PROMPT,
    state_schema=DeepAgentState,
)

editor_tools = [ls, read_file, write_file, cleanup_files]
editor_agent = create_agent(
    model=llm,
    tools=editor_tools,
    system_prompt=EDITOR_PROMPT,
    state_schema=DeepAgentState,
)


In [8]:
# -------------------------
# Orchestrator Tools (call other agents)
# -------------------------
from typing import Annotated
from langgraph.prebuilt import InjectedState
from langchain_core.tools import InjectedToolCallId
from langgraph.types import Command

@tool
def run_researcher(
    theme_id: int,
    thematic_question: str,
    state: Annotated[DeepAgentState, InjectedState],
    max_retries: int = 2
) -> str:
    """
    Run a single Research agent for ONE thematic question.

    Args:
        theme_id: The theme number (1, 2, 3, etc.)
        thematic_question: The specific thematic question to research
        state: Injected agent state providing user_id/thread_id
        max_retries: Number of retry attempts if researcher fails

    The Researcher will:
    - Receive ONE thematic question as input
    - Break it into 2-4 focused search queries
    - Use web_search to gather information
    - Write files to researcher/ folder with hash-based names:
        * researcher/<hash>_theme.md (research findings)
        * researcher/<hash>_sources.txt (raw sources)

    Returns a short status string for the orchestrator.
    """
    from file_tools import generate_hash
    
    # Generate hash for unique file naming
    file_hash = generate_hash(f"{theme_id}_{thematic_question}")
    
    # Build sub-state with theme-specific context
    sub_state: DeepAgentState = {
        "messages": state["messages"] + [
            AIMessage(content=f"[THEME {theme_id}] Research this question: {thematic_question}\n\n"
                             f"File hash: {file_hash}\n"
                             f"Save your findings to: researcher/{file_hash}_theme.md\n"
                             f"Save your sources to: researcher/{file_hash}_sources.txt")
        ],
        "user_id": state.get("user_id"),
        "thread_id": state.get("thread_id"),
    }
    
    # Execute researcher with retry logic
    for attempt in range(max_retries + 1):
        try:
            researcher_agent.invoke(sub_state)
            return f"✓ Theme {theme_id} research completed (hash: {file_hash})"
        except Exception as e:
            if attempt < max_retries:
                print(f"⚠ Theme {theme_id} failed (attempt {attempt + 1}/{max_retries + 1}), retrying...")
                continue
            else:
                return f"✗ Theme {theme_id} failed after {max_retries + 1} attempts: {str(e)}"
    
    return f"✓ Theme {theme_id} research completed (hash: {file_hash})"


In [9]:
@tool
def write_research_plan(
    thematic_questions: list[str],
    state: Annotated[DeepAgentState, InjectedState],
    tool_call_id: Annotated[str, InjectedToolCallId],
) -> Command:
    """
    Write the high-level research plan with major thematic questions.

    Args:
        thematic_questions: List of 3-5 major thematic questions that break down
                           the user's query. These guide the Researcher agent.
        state: Injected agent state providing user_id/thread_id.
        tool_call_id: Tool call ID for attaching a ToolMessage.

    The research_plan.md will be read by the Researcher to guide tactical research.

    Returns:
        Command with a ToolMessage confirming the plan was written.
    """
    from file_tools import _disk_path
    from langchain_core.messages import ToolMessage
    from langgraph.types import Command
    
    # Build content for research_plan.md
    content = "# Research Plan\n\n"
    content += "## User Query\n"
    # Extract user query from messages
    user_msg = [m for m in state["messages"] if hasattr(m, 'content')]
    if user_msg:
        content += f"{user_msg[-1].content}\n\n"
    
    content += "## Thematic Questions\n\n"
    for i, question in enumerate(thematic_questions, 1):
        content += f"{i}. {question}\n"
    
    # Write to disk
    path = _disk_path(state, "research_plan.md")
    with open(path, "w", encoding="utf-8") as f:
        f.write(content)
    
    msg = f"[RESEARCH PLAN WRITTEN] research_plan.md with {len(thematic_questions)} thematic questions -> {path}"
    return Command(
        update={
            "messages": [ToolMessage(msg, tool_call_id=tool_call_id)]
        }
    )


@tool
def run_editor(
    state: Annotated[DeepAgentState, InjectedState]
) -> str:
    """
    Run the Editor agent for this user/thread.

    The Editor will:
    - read research_plan.md, theme files, sources.txt
    - write the final answer to report.md

    Returns a short status string for the orchestrator.
    """
    sub_state: DeepAgentState = {
        "messages": [HumanMessage(content="Read research_plan.md and all files in the researcher/ folder, then synthesize everything into a comprehensive report.md file.")],
        "user_id": state.get("user_id"),
        "thread_id": state.get("thread_id"),
    }
    response = editor_agent.invoke(sub_state)
    # return "Editor completed. Final report is written to report.md."
    return f"Ideally report.md should be generated by now but check it with following response if it is generate othwise thell the issues. Here is editor agent response. Response:\n\n{response}"

In [10]:
# -------------------------
# Orchestrator Agent
# -------------------------

orchestrator_tools = [
    write_research_plan,  # strategic planning
    run_researcher,       # tactical research
    run_editor,           # synthesis
    cleanup_files,        # resetting workspace
]

orchestrator_agent = create_agent(
    model=llm,
    tools=orchestrator_tools,
    system_prompt=ORCHESTRATOR_PROMPT,
    state_schema=DeepAgentState,
)

In [11]:
# orchestrator_agent
# editor_agent


### Example usage


In [12]:
from langchain.messages import HumanMessage
state = {
        "messages": [HumanMessage(content="Tell me a joke about computers.")],
        "user_id": "user_123",
        "thread_id": "thread_mcp_001",
    }

result = orchestrator_agent.invoke(state)
result

{'messages': [HumanMessage(content='Tell me a joke about computers.', additional_kwargs={}, response_metadata={}, id='9af5bf9c-2c29-4609-86ed-dd7371ea1524'),
  AIMessage(content=[{'type': 'text', 'text': 'Why did the computer show up late to work?\n\nBecause it had a hard drive!', 'extras': {'signature': 'EqACCp0CAXLI2nyDbp73KlwBZ8K3LIyTiLqST5RDYhtuHHR7OzMPq5XRZuixWrDvj6bQuHtP16nYMJNKods+AqmbimcLAnl9GMQC8gPbb3prsvU+8+hFz4B93GoJte1isWGn8t+CLeTmvCv/Wv+zb/ptu5NI29myX3GSuOnG/AssQiU72lzCF5LTdrJJeGiKThou05NM0STU5LlVigz2igyIanWmzWLsK9/+DAN7bx8Qfd9p9aJE1eAqrG8XuNv4umedLaMuEItUuaHL8gAaraYlZOwiMB8GkjJhZzG53LuG4ch6K2PwgLMdiwSWMEX1Di1WGi+UIwcQd1voxK0qVCT+X7yjiBQ0on5u6l9dvpH5jZzAzYCE+mXJgl3QpOOqroTf'}}], additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-3-flash-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--a05cdd82-6f3b-49fb-a00f-c5f7ce0496a7-0', usage_metadata={'inpu

In [13]:
from langchain.messages import HumanMessage
state = {
        "messages": [HumanMessage(content="Do a detailed, well-structured analysis of MCP including history")],
        "user_id": "user_123",
        "thread_id": "thread_mcp_001",
    }

result = orchestrator_agent.invoke(state)

In [14]:
result

{'messages': [HumanMessage(content='Do a detailed, well-structured analysis of MCP including history', additional_kwargs={}, response_metadata={}, id='04fd0fdf-f4a7-4c2b-bcec-03f4840b122b'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'write_research_plan', 'arguments': '{"thematic_questions": ["What is MCP (Model Context Protocol) and what core problems does it solve in the AI ecosystem?", "What is the history, evolution, and origin of the Model Context Protocol?", "What are the key architectural components, protocols, and technical standards of MCP?", "What are the practical use cases, implementations, and ecosystem support for MCP?", "What are the benefits, limitations, and future outlook of MCP?"]}'}, '__gemini_function_call_thought_signatures__': {'7b5995f0-982e-4baa-a244-01983c44280c': 'ErQICrEIAXLI2nzcK6z75vjkbxK5Tq3dI9lg/MBZY1pVM1UYladsNih72kQGpriv78X65/h7yazOjtAAdD2xjeDGfMjSqVk76ZG8Zrc+9QKo1wmirkH1QhSIgTZtD1mh6+kiY3lbmsK+vEKgCG1u9kGokR1Vu9im8jSyh6CBLE0